# Day I2 — 보행의 기능 과제와 foot rocker: 몸은 무엇으로 앞으로 가는가

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I2.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I2.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 영상에서 phase별로 캡처할 프레임 정하기

같은 발의 연속 두 IC 프레임만 찾으면, 각 phase의 가운데에 해당하는 프레임 번호를 계산할 수 있습니다. 아래 프레임 번호는 **가상**입니다.

In [ ]:
import numpy as np

fps = 30                       # 휴대폰 영상의 초당 프레임 수(내 영상 정보에서 확인)
ic1, ic2 = 47, 81              # 같은 발(오른발)의 연속 두 initial contact가 보인 프레임 번호(가상)
stride_frames = ic2 - ic1
print("stride = %d frames = %.2f s, 한 프레임 = %.1f ms = %.1f%%GC"
      % (stride_frames, stride_frames / fps, 1000 / fps, 100 / stride_frames))

# Perry phase(시작 %GC), 과제, 그 phase의 rocker(Perry 4개 / Whittle 3개)
plan = [("IC",   0, "WA",  "heel rocker 시작",            "heel (initial) rocker"),
        ("LR",   2, "WA",  "heel rocker",                 "heel (initial) rocker"),
        ("MSt", 12, "SLS", "ankle rocker",                "ankle (second) rocker"),
        ("TSt", 31, "SLS", "forefoot rocker",             "forefoot (terminal) rocker"),
        ("PSw", 50, "SLA", "toe rocker",                  "forefoot (terminal) rocker에 포함"),
        ("ISw", 62, "SLA", "-", "-"),
        ("MSw", 75, "SLA", "-", "-"),
        ("TSw", 87, "SLA", "-", "-")]
print("\nphase  시작%GC  캡처할 프레임(가운데)  과제  Perry rocker        Whittle rocker")
ends = [p[1] for p in plan[1:]] + [100]
for (name, start, task, perry, whittle), end in zip(plan, ends):
    mid = ic1 + (start + end) / 2 / 100 * stride_frames
    print("%-5s  %4d      %6.1f → %d번            %-4s  %-18s  %s" % (name, start, mid, round(mid), task, perry, whittle))

### 예시 3 — 옆에서 찍은 무릎 각도가 작게 보이는 이유

카메라 한 대는 3차원의 다리를 2차원 화면에 눌러 담습니다. 다리가 카메라와 평행한 면에서 벗어나 돌아가 있으면 정강이가 짧아 보이고, 굽힘 각도가 실제보다 작게 찍힙니다. Perry는 실제 60°인 swing 무릎 굽힘이 다리가 안쪽으로 돌아가면 40°쯤으로 보일 수 있고, 그러면 "무릎이 덜 굽는다"고 잘못 판단할 수 있다는 예를 듭니다([P] 20장 도입). 아래는 허벅지를 수직으로 둔 아주 단순한 모형으로 이 효과를 계산한 것입니다(모형과 식은 일반적인 기하로 새로 만든 것이고 책에 있는 식이 아닙니다).

In [ ]:
import numpy as np

# 단순 모형: 허벅지는 수직, 무릎은 실제로 60° 굽어 있다.
# 다리 전체가 수직축을 중심으로 alpha만큼 돌아 카메라 평면에서 벗어나면,
# 옆 카메라 화면에는 정강이가 짧아 보이고 무릎 각도가 작게 찍힌다.
true_knee = 60.0
for alpha in [0, 15, 30, 45, 60]:
    a, k = np.radians(alpha), np.radians(true_knee)
    shank = np.array([np.sin(k) * np.cos(a),   # 화면 가로(진행 방향) 성분
                      np.sin(k) * np.sin(a),   # 카메라 쪽 깊이 성분: 화면에서 사라진다
                      -np.cos(k)])             # 화면 세로 성분
    seen = np.degrees(np.arctan2(shank[0], -shank[2]))   # 화면에 투영된 무릎 각도
    print("다리 회전 %2d° → 화면의 무릎 굽힘 %4.1f° (실제 %.0f°, 차이 %5.1f°)"
          % (alpha, seen, true_knee, seen - true_knee))